# Retrieval ablation experiments — AIC 2026 Round 3

Notebook này gọi backend qua HTTP, không import trực tiếp `RetrievalService`. Ground truth chỉ được dùng để đánh giá exact frame và same-video retrieval. Mỗi lần chạy tạo một thư mục kết quả có timestamp trong `data/experiments/results/`.

**Quy trình:** chạy cell cấu hình, kiểm tra kết nối/load ground truth, rồi bật một `RUN_*` flag cho từng experiment. Cell temporal mặc định chọn 10 truy vấn KIS.

In [1]:
from __future__ import annotations

import csv
import json
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests
import yaml
from IPython.display import HTML, Image, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'apps').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise RuntimeError('Không tìm thấy repository root. Mở notebook từ trong repository.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
GROUND_TRUTH_PATH = REPO_ROOT / 'data/experiments/aic_2026_groundtruth/aic2026_round_3.csv'
AGENT_CONFIG_PATH = REPO_ROOT / 'configs/agent.yaml'
SERVER_BASE_URL = 'http://localhost:8000'  # Ví dụ remote: http://gpu-server:8000
SEARCH_ENDPOINT = f"{SERVER_BASE_URL.rstrip('/')}/api/retrieval/search"
PLAN_ENDPOINT = f"{SERVER_BASE_URL.rstrip('/')}/api/retrieval/plan"
PROFILE = 'competition_default'
TOP_K = 50
REQUEST_TIMEOUT_S = 180
KIS_LIMIT = 10

RUN_LABEL = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULTS_DIR = REPO_ROOT / 'data/experiments/results' / f'retrieval_round3_{RUN_LABEL}'
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
HTTP = requests.Session()

print(f'Repository : {REPO_ROOT}')
print(f'Ground truth: {GROUND_TRUTH_PATH}')
print(f'Results    : {RESULTS_DIR}')
print(f'Server     : {SERVER_BASE_URL}')

Repository : D:\University\Projects\Individual projects\Multimodal-Retrieval
Ground truth: D:\University\Projects\Individual projects\Multimodal-Retrieval\data\experiments\aic_2026_groundtruth\aic2026_round_3.csv
Results    : D:\University\Projects\Individual projects\Multimodal-Retrieval\data\experiments\results\retrieval_round3_20260915T170835Z
Server     : http://localhost:8000


In [5]:
# Shared helpers: CSV parsing, HTTP calls, evaluation, visualization, and result persistence.

QUERY_ID_RE = re.compile(r'query-(p\d+-\d+)-(kis|qa|trake)', re.IGNORECASE)
TARGET_RE = re.compile(r'(L\d+_(?:V)?\d+)\s*,\s*(\d+)', re.IGNORECASE)


def check_server() -> None:
    response = HTTP.get(f"{SERVER_BASE_URL.rstrip('/')}/docs", timeout=10)
    response.raise_for_status()
    print(f'Server is reachable: HTTP {response.status_code}')


def query_id_and_type(raw_query: str) -> tuple[str, str]:
    match = QUERY_ID_RE.search(raw_query or '')
    if not match:
        raise ValueError(f'Không đọc được query id/type từ: {raw_query[:80]!r}')
    return match.group(1).lower(), match.group(2).upper()


def retrieval_text(raw_query: str) -> str:
    lines = [line.strip() for line in (raw_query or '').splitlines() if line.strip()]
    return '\n'.join(lines[1:]).strip() if lines and QUERY_ID_RE.search(lines[0]) else '\n'.join(lines).strip()


def parse_targets(answer: str) -> list[tuple[str, int]]:
    # Some rows contain several valid video-frame pairs; preserve all of them for Recall@K.
    targets = [(match.group(1).upper(), int(match.group(2))) for match in TARGET_RE.finditer(str(answer))]
    if not targets:
        raise ValueError(f'Ground-truth answer needs at least one video,frame pair: {answer!r}')
    return targets


def load_round3(path: Path = GROUND_TRUTH_PATH) -> pd.DataFrame:
    with path.open(encoding='utf-8-sig', newline='') as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError(f'Ground truth is empty: {path}')
    columns = list(rows[0])
    if len(columns) < 3 or columns[0] != 'Query':
        raise ValueError(f'Unexpected CSV columns: {columns}')
    answer_column = columns[2]  # Do not rely on a locale-specific header spelling.
    records: list[dict[str, Any]] = []
    for row in rows:
        raw_query = row['Query']
        query_id, query_type = query_id_and_type(raw_query)
        raw_answer = str(row[answer_column] or '').strip()
        has_ground_truth = bool(raw_answer)
        targets = parse_targets(raw_answer) if has_ground_truth else []
        target_video_code, target_frame_idx = targets[0] if targets else (None, None)
        records.append({
            'query_id': query_id,
            'query_type': query_type,
            'query_text_vi': retrieval_text(raw_query),
            'target_video_code': target_video_code,
            'target_frame_idx': target_frame_idx,
            'has_ground_truth': has_ground_truth,
            'ground_truth_targets': targets,
            'target_count': len(targets),
            'ground_truth_raw': raw_answer,
        })
    return pd.DataFrame(records)


def call_retrieval(query_text: str, query_type: str = 'KIS', *, top_k: int = TOP_K,
                   profile: str = PROFILE, options: dict[str, Any] | None = None,
                   query_name: str | None = None) -> dict[str, Any]:
    payload = {
        'query_name': query_name,
        'query_type': query_type,
        'query_text': query_text,
        'top_k': top_k,
        'profile': profile,
        'options': options or {},
    }
    response = HTTP.post(SEARCH_ENDPOINT, json=payload, timeout=REQUEST_TIMEOUT_S)
    if not response.ok:
        raise RuntimeError(f'Retrieval failed ({response.status_code}): {response.text[:1000]}')
    return response.json()


def rank_metrics(response: dict[str, Any], targets: list[tuple[str, int]]) -> dict[str, Any]:
    if not targets:
        return {'exact_frame_rank': None, 'same_video_rank': None, 'exact_hit_at_k': None, 'same_video_hit_at_k': None}
    exact_frame_rank = None
    same_video_rank = None
    for result in response.get('results', []):
        rank = int(result['rank'])
        if result.get('video_code') in {video for video, _ in targets} and same_video_rank is None:
            same_video_rank = rank
        if ((result.get('video_code'), result.get('frame_idx')) in set(targets)
                and exact_frame_rank is None):
            exact_frame_rank = rank
    return {
        'exact_frame_rank': exact_frame_rank,
        'same_video_rank': same_video_rank,
        'exact_hit_at_k': exact_frame_rank is not None,
        'same_video_hit_at_k': same_video_rank is not None,
    }


def result_frame(response: dict[str, Any], targets: list[tuple[str, int]]) -> pd.DataFrame:
    rows = []
    for item in response.get('results', []):
        rows.append({
            'rank': item.get('rank'),
            'score': item.get('score'),
            'video_code': item.get('video_code'),
            'frame_idx': item.get('frame_idx'),
            'timestamp_ms': item.get('timestamp_ms'),
            'exact_gt': (item.get('video_code'), item.get('frame_idx')) in set(targets),
            'same_video_gt': item.get('video_code') in {video for video, _ in targets},
            'image_url': item.get('image_url'),
        })
    return pd.DataFrame(rows)


def image_url(url: str | None) -> str | None:
    if not url:
        return None
    return url if url.startswith(('http://', 'https://')) else f"{SERVER_BASE_URL.rstrip('/')}{url}"


def show_results(response: dict[str, Any], targets: list[tuple[str, int]], *, max_images: int = 10) -> pd.DataFrame:
    table = result_frame(response, targets)
    display(table.drop(columns=['image_url'], errors='ignore'))
    for _, row in table.head(max_images).iterrows():
        marker = ' ★ EXACT GT' if row['exact_gt'] else (' • same video' if row['same_video_gt'] else '')
        display(HTML(f"<b>Rank {row['rank']} | {row['video_code']} | frame {row['frame_idx']} | score {row['score']:.4f}{marker}</b>"))
        url = image_url(row.get('image_url'))
        if url:
            display(Image(url=url, width=320))
    return table


def save_experiment(name: str, records: list[dict[str, Any]], responses: dict[str, Any]) -> Path:
    stem = RESULTS_DIR / name
    pd.DataFrame(records).to_csv(stem.with_suffix('.csv'), index=False, encoding='utf-8-sig')
    stem.with_suffix('.json').write_text(json.dumps(responses, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved: {stem}.csv and {stem}.json')
    return stem


In [6]:
check_server()
round3 = load_round3()
evaluated_round3 = round3.loc[round3['has_ground_truth']].copy()
kis_round3 = evaluated_round3.loc[evaluated_round3['query_type'].eq('KIS')].copy()
print(f'All Round 3 queries: {len(round3)} | evaluable: {len(evaluated_round3)} | evaluable KIS: {len(kis_round3)}')
display(round3[['query_id', 'query_type', 'target_video_code', 'target_frame_idx']].head(12))
assert len(kis_round3) >= KIS_LIMIT, 'CSV does not contain enough KIS rows with ground truth for the configured experiment.'

Server is reachable: HTTP 200
All Round 3 queries: 36 | evaluable: 35 | evaluable KIS: 25


,query_id,query_type,target_video_code,target_frame_idx
0,p2-1,KIS,L26_V183,5991.0
1,p2-2,KIS,L22_V026,1514.0
2,p2-3,KIS,L26_V390,6010.0
3,p2-4,KIS,L25_V075,975.0
4,p2-5,QA,L25_V016,3651.0
5,p2-6,QA,L25_V027,11842.0
6,p2-7,KIS,L22_V024,20783.0
7,p2-8,QA,L21_V018,21393.0
8,p2-9,KIS,NaN,NaN
9,p2-10,KIS,L21_V003,19779.0


## 1. Tối ưu câu query — Vietnamese vs English, SigLIP2 cố định

Điền bản dịch English thủ công hoặc từ phương pháp dịch bạn muốn đánh giá. Hai variant dùng cùng ground truth, profile và SigLIP2; agent planning/query expansion bị tắt để đo tác động của wording thuần túy.

In [4]:
# --- Editable query-optimization configuration ---
RUN_QUERY_OPTIMIZATION = False
QUERY_ROW_ID = 'p2-1'
QUERY_VI_OVERRIDE = None  # None = original Vietnamese from CSV
QUERY_EN = 'A chef pours beaten egg into a soup pot with mushrooms and bamboo shoots, then cuts tofu above the pot and adds it before stirring.'
TARGET_VIDEO_CODE_OVERRIDE = None  # Example: 'L26_V183'
TARGET_FRAME_IDX_OVERRIDE = None   # Example: 5991
QUERY_OPT_TOP_K = TOP_K

if RUN_QUERY_OPTIMIZATION:
    selected = round3.loc[round3['query_id'].eq(QUERY_ROW_ID)].iloc[0]
    targets = ([(TARGET_VIDEO_CODE_OVERRIDE or selected.target_video_code, TARGET_FRAME_IDX_OVERRIDE or int(selected.target_frame_idx))]
               if TARGET_VIDEO_CODE_OVERRIDE or TARGET_FRAME_IDX_OVERRIDE else selected.ground_truth_targets)
    variants = {
        'vietnamese_original': QUERY_VI_OVERRIDE or selected.query_text_vi,
        'english_translation': QUERY_EN,
    }
    records, responses = [], {}
    fixed_options = {
        'visual_search_mode': 'siglip2',
        'use_query_expansion': False,
        'use_agent_query_planning': False,
        'use_reranker': False,
        'temporal_mode': False,
    }
    for variant_name, query_text in variants.items():
        if not query_text or not query_text.strip():
            raise ValueError(f'Missing text for {variant_name}')
        response = call_retrieval(query_text, selected.query_type, top_k=QUERY_OPT_TOP_K, options=fixed_options,
                                  query_name=f'round3-{QUERY_ROW_ID}-{variant_name}')
        responses[variant_name] = response
        records.append({'query_id': QUERY_ROW_ID, 'variant': variant_name, 'query_text': query_text,
                        'visual_search_mode': 'siglip2', **rank_metrics(response, targets)})
    display(pd.DataFrame(records))
    for variant_name, response in responses.items():
        display(HTML(f'<h4>{variant_name}</h4>'))
        show_results(response, targets)
    save_experiment(f'query_optimization_{QUERY_ROW_ID}', records, responses)
else:
    print('Set RUN_QUERY_OPTIMIZATION = True, then run this cell.')

Set RUN_QUERY_OPTIMIZATION = True, then run this cell.


## 2. Ablation embedding model — OpenCLIP, SigLIP2, both

`visual_search_mode` map trực tiếp tới API contract. Giữ nguyên query/options còn lại để so sánh công bằng.

In [5]:
# --- Editable embedding-ablation configuration ---
RUN_EMBEDDING_ABLATION = False
EMBEDDING_QUERY_ROW_ID = 'p2-1'
EMBEDDING_QUERY_OVERRIDE = None
EMBEDDING_MODES = ['openclip', 'siglip2', 'both']
EMBEDDING_TOP_K = TOP_K

if RUN_EMBEDDING_ABLATION:
    selected = round3.loc[round3['query_id'].eq(EMBEDDING_QUERY_ROW_ID)].iloc[0]
    query_text = EMBEDDING_QUERY_OVERRIDE or selected.query_text_vi
    records, responses = [], {}
    for visual_mode in EMBEDDING_MODES:
        response = call_retrieval(query_text, selected.query_type, top_k=EMBEDDING_TOP_K,
                                  options={'visual_search_mode': visual_mode, 'use_query_expansion': False,
                                           'use_agent_query_planning': False, 'use_reranker': False, 'temporal_mode': False},
                                  query_name=f'round3-{EMBEDDING_QUERY_ROW_ID}-{visual_mode}')
        responses[visual_mode] = response
        records.append({'query_id': EMBEDDING_QUERY_ROW_ID, 'visual_search_mode': visual_mode,
                        **rank_metrics(response, selected.ground_truth_targets)})
    display(pd.DataFrame(records))
    for visual_mode, response in responses.items():
        display(HTML(f'<h4>{visual_mode}</h4>'))
        show_results(response, selected.ground_truth_targets)
    save_experiment(f'embedding_ablation_{EMBEDDING_QUERY_ROW_ID}', records, responses)
else:
    print('Set RUN_EMBEDDING_ABLATION = True, then run this cell.')

Set RUN_EMBEDDING_ABLATION = True, then run this cell.


## 3. Temporal KIS strategy ablation — ATS, Vortex, DEV-first

Mỗi strategy chạy cùng 10 KIS query đầu tiên theo thứ tự CSV, với cùng `top_k`, profile và visual mode. Bảng summary báo exact-frame và same-video Recall@K/MRR; bên dưới hiển thị top-K của một query/strategy được chọn.

In [6]:
# --- Editable temporal-ablation configuration ---
RUN_TEMPORAL_ABLATION = False
TEMPORAL_QUERY_IDS = kis_round3['query_id'].head(KIS_LIMIT).tolist()  # Or specify exactly 10 IDs.
TEMPORAL_STRATEGIES = ['aithena_weighted_ats', 'vortex_k_context', 'dev_first_search']
TEMPORAL_VISUAL_MODE = 'both'  # profile | openclip | siglip2 | both
TEMPORAL_TOP_K = TOP_K
TEMPORAL_USE_AGENT_PLANNING = True
TEMPORAL_USE_QUERY_EXPANSION = False  # Server bypasses standalone expansion for temporal KIS.
DISPLAY_TEMPORAL_QUERY_ID = TEMPORAL_QUERY_IDS[0]
DISPLAY_TEMPORAL_STRATEGY = 'dev_first_search'

if RUN_TEMPORAL_ABLATION:
    selected_queries = kis_round3.loc[kis_round3['query_id'].isin(TEMPORAL_QUERY_IDS)].copy()
    if len(selected_queries) != len(TEMPORAL_QUERY_IDS):
        raise ValueError('At least one TEMPORAL_QUERY_ID is not a KIS query in Round 3.')
    records, responses = [], {}
    for _, item in selected_queries.iterrows():
        for strategy in TEMPORAL_STRATEGIES:
            options = {
                'temporal_mode': True,
                'temporal_strategy': strategy,
                'visual_search_mode': TEMPORAL_VISUAL_MODE,
                'use_agent_query_planning': TEMPORAL_USE_AGENT_PLANNING,
                'use_query_expansion': TEMPORAL_USE_QUERY_EXPANSION,
            }
            response = call_retrieval(item.query_text_vi, 'KIS', top_k=TEMPORAL_TOP_K, options=options,
                                      query_name=f'round3-{item.query_id}-{strategy}')
            key = f'{item.query_id}__{strategy}'
            responses[key] = response
            records.append({'query_id': item.query_id, 'strategy': strategy,
                            'target_video_code': item.target_video_code, 'target_frame_idx': int(item.target_frame_idx),
                            'top_k': TEMPORAL_TOP_K, 'visual_search_mode': TEMPORAL_VISUAL_MODE,
                            **rank_metrics(response, item.ground_truth_targets)})
    results = pd.DataFrame(records)
    summary = results.groupby('strategy', as_index=False).agg(
        queries=('query_id', 'count'),
        exact_recall_at_k=('exact_hit_at_k', 'mean'),
        same_video_recall_at_k=('same_video_hit_at_k', 'mean'),
    )
    summary['exact_mrr_at_k'] = summary['strategy'].map(
        results.assign(rr=results.exact_frame_rank.map(lambda rank: 0.0 if pd.isna(rank) else 1.0 / rank)).groupby('strategy')['rr'].mean()
    )
    display(summary.sort_values('strategy'))
    target = selected_queries.loc[selected_queries['query_id'].eq(DISPLAY_TEMPORAL_QUERY_ID)].iloc[0]
    display(HTML(f'<h4>Top-{TEMPORAL_TOP_K}: {DISPLAY_TEMPORAL_QUERY_ID} / {DISPLAY_TEMPORAL_STRATEGY}</h4>'))
    show_results(responses[f'{DISPLAY_TEMPORAL_QUERY_ID}__{DISPLAY_TEMPORAL_STRATEGY}'],
                 target.ground_truth_targets)
    save_experiment('temporal_kis_ablation', records, responses)
else:
    print('Set RUN_TEMPORAL_ABLATION = True, then run this cell. It will submit len(TEMPORAL_QUERY_IDS) × len(TEMPORAL_STRATEGIES) requests.')

Set RUN_TEMPORAL_ABLATION = True, then run this cell. It will submit len(TEMPORAL_QUERY_IDS) × len(TEMPORAL_STRATEGIES) requests.


## 4. Prompt-version experiment cho agent planner

`execution_mode: direct` hiện tại chỉ sử dụng `agents.planner.system_prompt`; prompt của sub-agent decomposition/expansion chỉ có hiệu lực khi chuyển server sang `deep_agent`. Để tránh ghi đè vô ý, việc sửa `configs/agent.yaml` là opt-in và luôn sao lưu file gốc vào thư mục results. Server Docker mount config dạng read-only ở phía container, nhưng host-side update vẫn được container đọc ở request kế tiếp.

In [7]:
# --- Editable prompt-version configuration ---
RUN_PROMPT_EXPERIMENT = False
APPLY_PROMPT_TO_CONFIG = False  # Set True only when this notebook and backend share this repository/config file.
PROMPT_VERSION = 'agent_v2'  # Must match llm_query_planning.prompt_versions.versions in configs/agent.yaml
PROMPT_QUERY_ROW_ID = 'p2-1'
PROMPT_TOP_K = TOP_K

agent_config = yaml.safe_load(AGENT_CONFIG_PATH.read_text(encoding='utf-8'))
planning_config = agent_config['llm_query_planning']
PROMPT_VERSIONS = planning_config.get('prompt_versions', {}).get('versions', {})
print(f"Configured prompt versions: {list(PROMPT_VERSIONS)} | active: {planning_config.get('prompt_versions', {}).get('active_version', 'legacy')}")


def apply_planner_prompt(version: str) -> Path:
    if version not in PROMPT_VERSIONS:
        raise KeyError(f'Unknown prompt version {version!r}. Available: {list(PROMPT_VERSIONS)}')
    if not APPLY_PROMPT_TO_CONFIG:
        raise RuntimeError('Set APPLY_PROMPT_TO_CONFIG = True before changing configs/agent.yaml.')
    backup_path = RESULTS_DIR / f'agent_before_{version}.yaml'
    shutil.copy2(AGENT_CONFIG_PATH, backup_path)
    updated = yaml.safe_load(AGENT_CONFIG_PATH.read_text(encoding='utf-8'))
    planning = updated['llm_query_planning']
    versions = planning.get('prompt_versions', {}).get('versions', {})
    if version not in versions:
        raise KeyError(f'Unknown config prompt version {version!r}. Add it under llm_query_planning.prompt_versions.versions first.')
    if planning.get('execution_mode') != 'direct':
        print('Warning: deep_agent mode may delegate selectively; the root planner still uses this version.')
    planning['prompt_versions']['active_version'] = version
    AGENT_CONFIG_PATH.write_text(yaml.safe_dump(updated, allow_unicode=True, sort_keys=False), encoding='utf-8')
    print(f'Applied {version}; backup saved at {backup_path}')
    return backup_path


if RUN_PROMPT_EXPERIMENT:
    backup_path = apply_planner_prompt(PROMPT_VERSION)
    selected = round3.loc[round3['query_id'].eq(PROMPT_QUERY_ROW_ID)].iloc[0]
    options = {'temporal_mode': selected.query_type == 'KIS', 'temporal_strategy': 'dev_first_search',
               'visual_search_mode': 'both', 'use_agent_query_planning': True, 'use_query_expansion': False}
    response = call_retrieval(selected.query_text_vi, selected.query_type, top_k=PROMPT_TOP_K, options=options,
                              query_name=f'round3-{selected.query_id}-prompt-{PROMPT_VERSION}')
    record = {'query_id': selected.query_id, 'prompt_version': PROMPT_VERSION, 'backup_path': str(backup_path),
              **rank_metrics(response, selected.ground_truth_targets)}
    display(pd.DataFrame([record]))
    display(HTML('<h4>Planner output used by retrieval</h4>'))
    display(response.get('normalized_query', {}))
    show_results(response, selected.ground_truth_targets)
    save_experiment(f'prompt_{PROMPT_VERSION}_{selected.query_id}', [record], {PROMPT_VERSION: response})
else:
    print('Add/edit versions in configs/agent.yaml; then set both APPLY_PROMPT_TO_CONFIG and RUN_PROMPT_EXPERIMENT to True.')

Configured prompt versions: ['agent_v1', 'agent_v2'] | active: agent_v1
Add/edit versions in configs/agent.yaml; then set both APPLY_PROMPT_TO_CONFIG and RUN_PROMPT_EXPERIMENT to True.


In [8]:
# Optional safety cell: restore a backup deliberately after a prompt experiment.
RESTORE_AGENT_CONFIG_FROM = None  # Example: RESULTS_DIR / 'agent_before_visual_precision_v1.yaml'

if RESTORE_AGENT_CONFIG_FROM:
    source = Path(RESTORE_AGENT_CONFIG_FROM)
    if not source.is_file():
        raise FileNotFoundError(source)
    shutil.copy2(source, AGENT_CONFIG_PATH)
    print(f'Restored {AGENT_CONFIG_PATH} from {source}')
else:
    print('No config restore requested.')

No config restore requested.
